In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
sns.set_style('ticks')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 25
two_colors = ['#6384B3', '#F08743']


In [ ]:
with open('./processed_data/heatmap_em_new.pkl','rb') as f:
    mat = pickle.load(f)

In [ ]:
avg = 0.374
min_value = 0.232
max_value=0.569

In [ ]:
plt.figure(figsize=(15,15))
sns.heatmap(mat, center=avg, vmin=min_value, vmax=max_value, cmap=sns.color_palette("vlag_r", as_cmap=True), fmt='.2f',annot=True,
            cbar_kws={'label':'Propensity for Deep ICs'},square=True,linewidths=0.5, linecolor='gray').invert_yaxis()
# plt.xticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
# plt.yticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
plt.xticks(np.arange(0,15,5),['{}%'.format((i)*10) for i in np.arange(0,15,5)])
plt.yticks(np.arange(5,15,5),['{}%'.format((i)*10) for i in np.arange(5,15,5)])
plt.ylabel('Degree of Exploration')
plt.xlabel('Degree of Exploitation')

# Get the current axes and show the outer spines.
ax = plt.gca()

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1)  # Set the spine width.


In [ ]:
from sklearn.linear_model import LinearRegression

df = []
for i in range(0,10):
    for j in range(0,10):
       df.append((i+1,j+1, mat[i,j]))
df = pd.DataFrame(df, columns=['x1', 'x2', 'y'])

# Define the feature matrix X and target y.
X = df[['x1', 'x2']]
y = df['y']

# Create the linear regression model.
model = LinearRegression()

# Fit the model.
model.fit(X, y)

# Extract fitted parameters k_1 and k_2.
k_1, k_2 = model.coef_
intercept = model.intercept_
r_squared = model.score(X, y)

print(f"Fitted equation: y = {intercept} + {k_1} * x1 + {k_2} * x2")
print(f"R-squared: {r_squared}")

In [ ]:
mat_predicted = np.zeros((10,10))
for i in range(0,10):
    for j in range(0,10):
       mat_predicted[i,j] = model.predict([[i+1,j+1]])



In [ ]:
plt.figure(figsize=(15,15))
sns.heatmap(mat_predicted, center=avg, vmin=min_value, vmax=max_value, cmap=sns.color_palette("vlag_r", as_cmap=True), fmt='.2f',annot=True,
            cbar_kws={'label':'Ratio for Deep ICs'},square=True,linewidths=0.5, linecolor='gray').invert_yaxis()
# plt.xticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
# plt.yticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
plt.xticks(np.arange(0,15,5),['{}%'.format((i)*10) for i in np.arange(0,15,5)])
plt.yticks(np.arange(5,15,5),['{}%'.format((i)*10) for i in np.arange(5,15,5)])
plt.ylabel('Degree of Exploration')

In [ ]:
with open('./processed_data/heatmap_model.pkl', 'rb') as f:
    results = pickle.load(f)
print(results)
plt.figure(figsize=(8,6))
sns.heatmap(results['mat'], center=avg, vmin=min_value, vmax=max_value, cmap=sns.color_palette("vlag_r", as_cmap=True),square=True,
            cbar_kws={'label':'Ratio for Deep ICs'},linewidths=0.5, linecolor='gray').invert_yaxis()
# plt.xticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
# plt.yticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
plt.xticks(np.arange(0,15,5),['{}%'.format((i)*10) for i in np.arange(0,15,5)])
plt.yticks(np.arange(5,15,5),['{}%'.format((i)*10) for i in np.arange(5,15,5)])
plt.ylabel('Degree of Exploration')
plt.xlabel('Degree of Exploitation')



# Get the current axes and show the outer spines.
ax = plt.gca()

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1)  # Set the spine width.

plt.savefig('./figures/figure3/heatmap_model.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)


In [ ]:
mat.shape

In [ ]:
results['mat'].shape

In [ ]:
results_df = []
for i in range(10):
    for j in range(10):
        results_df.append((mat[i,j],results['mat'][i,j]))
results_df = pd.DataFrame(results_df, columns=['empirical', 'model'])

In [ ]:
plt.figure(figsize=(6.5,6))
sns.regplot(results_df, x='empirical', y='model')
from sklearn.metrics import r2_score
r2 = r2_score(y_true=results_df['empirical'], y_pred=results_df['model'])
plt.xticks([0.2,0.3, 0.4, 0.5,0.6])
plt.yticks([0.2,0.3, 0.4, 0.5,0.6])
print(f"R^2: {r2}")
plt.ylabel('Model')
plt.xlabel('Emprical')
plt.text(0.22, 0.55, r'$R^2=0.90$')
plt.savefig('./figures/figure3/model_performance.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
#Load the previously simulated population sample.

def bernoulli_ci(values):
    import statsmodels.api as sm
    n = len(values)
    p = values.mean()
    conf_int = sm.stats.proportion_confint(p * n, n, method='normal')
    return conf_int
data = pd.read_pickle('./processed_data/samples_10000_model.pkl')

# Plot the distribution of inequality across groups.
plt.figure(figsize=(8,6))
sns.barplot(data, x='age_range', y='indicator', order=['18-23','24-30','31-40','41-50','51-60','60+'], errorbar=bernoulli_ci, palette=sns.color_palette("Blues", 6), capsize=0.2, edgecolor=sns.color_palette("Blues", 6))
plt.hlines(y=data['indicator'].mean(), xmin=-0.75, xmax=5.75, lw=5, color='gray', ls='--', alpha=0.5, label='Avg')
plt.ylabel('Ratio for Deep ICs')
plt.xlabel('Age')
plt.xlim(-0.75, 5.75)
plt.legend()
plt.savefig('./figures/figure3/IC_disparity_model.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
print(data['indicator'].mean())

In [ ]:
# Persistence

df = pd.read_pickle('./processed_data/samples_10000_model.pkl')

plt.figure(figsize=(4,4))
n_1, bins_1 = np.histogram(df.loc[df['indicator']==1]['timeInDEEP'], bins=100, density=True)
n_0, bins_0 = np.histogram(df.loc[df['indicator']==0]['timeInDEEP'], bins=100, density=True)
n_1 = n_1/np.sum(n_1)*100
n_0 = n_0/np.sum(n_0)*100
bins_0 = (bins_0[1:]+bins_0[:-1])/2
bins_1 = (bins_1[1:]+bins_1[:-1])/2
n_1 = np.cumsum(n_1)/100
n_0 = np.cumsum(n_0)/100
n_0 = np.concatenate([np.array([0]),n_0])
bins_0 = np.concatenate([np.array([0]),bins_0])
n_1 = np.concatenate([np.array([0]),n_1])
bins_1 = np.concatenate([np.array([0]),bins_1])

n_1 = 1- n_1
n_0 = 1- n_0
plt.plot(bins_1, n_1, lw=2, color='#6384B3')
plt.plot(bins_0, n_0, lw=2, color='#F08743')

# print(np.argmin(np.abs(n_1-0.5)))
# plt.hlines(xmin=-0.1, xmax=bins_1[89], y=n_1[89], color='r', ls='--')
# plt.vlines(ymin=-0.1, x=bins_1[89], ymax=n_1[89], color='r', ls='--')
# plt.scatter(x=bins_1[89], y=n_1[89],s=40,  marker='o', color='r', zorder=20)
# plt.text(x=bins_1[89]-0.2, y=n_1[89]+0.2, s='{:.2f}%'.format(bins_1[89]*100),fontsize=20)


# plt.xticks([0, 0.5, 1], ['0', '50%', '100%'])
# plt.xlim(-0.1, 1.1)
# plt.ylim(-0.1, 1.1)
plt.xlabel('Prop. of time in ICs')
plt.ylabel('CCDF')
plt.savefig('./figures/figure3/length_ccdf.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
df

In [ ]:
# length 
plt.figure(figsize=(6.5,6))
sns.barplot(x=df['indicator'], y=df['timeInDEEP'], order=[1.0,0.0], capsize=0.2, width=0.5, palette=two_colors, saturation=1, alpha=1)
plt.xlabel('')
plt.xticks([0,1],['Deep ICs','Others'])
plt.yticks([0, 0.2, 0.4,0.6, 0.8], ['{:.0f}%'.format(100*t) for t in [0, 0.2, 0.4,0.6, 0.8] ])
plt.ylabel(r'Proportion of time in ICs')
plt.xlim(-0.5, 1.5)
plt.savefig('./figures/figure3/length_bars.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
# Persistence

df = pd.read_pickle('./processed_data/samples_10000_model.pkl')
df['depth'] = 1-df['depth']
plt.figure(figsize=(4,4))
n_1, bins_1 = np.histogram(df.loc[df['indicator']==1]['depth'], bins=np.linspace(df['depth'].min(), df['depth'].max(), 100), density=True)
n_0, bins_0 = np.histogram(df.loc[df['indicator']==0]['depth'], bins=np.linspace(df['depth'].min(), df['depth'].max(), 100), density=True)
n_1 = n_1/np.sum(n_1)*100
n_0 = n_0/np.sum(n_0)*100
bins_0 = (bins_0[1:]+bins_0[:-1])/2
bins_1 = (bins_1[1:]+bins_1[:-1])/2
n_1 = np.cumsum(n_1)/100
n_0 = np.cumsum(n_0)/100
# n_0 = np.concatenate([np.array([0]),n_0])
# bins_0 = np.concatenate([np.array([0]),bins_0])
# n_1 = np.concatenate([np.array([0]),n_1])
# bins_1 = np.concatenate([np.array([0]),bins_1])

n_1 = 1- n_1
n_0 = 1- n_0
plt.plot(bins_1, n_1, lw=2, color='#6384B3')
plt.plot(bins_0, n_0, lw=2, color='#F08743')

# print(np.argmin(np.abs(n_1-0.5)))
# plt.hlines(xmin=-0.1, xmax=bins_1[89], y=n_1[89], color='r', ls='--')
# plt.vlines(ymin=-0.1, x=bins_1[89], ymax=n_1[89], color='r', ls='--')
# plt.scatter(x=bins_1[89], y=n_1[89],s=40,  marker='o', color='r', zorder=20)
# plt.text(x=bins_1[89]-0.2, y=n_1[89]+0.2, s='{:.2f}%'.format(bins_1[89]*100),fontsize=20)


# plt.xticks([0, 0.5, 1], ['0', '50%', '100%'])
# plt.xlim(-0.1, 1.1)
# plt.ylim(-0.1, 1.1)
plt.xlabel('Depth of ICs')
plt.ylabel('CCDF')
plt.savefig('./figures/figure3/depth_ccdf.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
# length 
plt.figure(figsize=(6.5,6))
sns.barplot(x=df['indicator'], y=df['depth'], order=[1.0,0.0], capsize=0.2, width=0.5, palette=two_colors, saturation=1, alpha=1)
plt.xlabel('')
plt.xticks([0,1],['Deep ICs','Others'])
plt.hlines(y=0, xmin=-0.5, xmax=1.5, color='k', lw=0.5)
plt.xlim(-0.5, 1.5)
plt.ylabel(r'Depth of ICs')
plt.savefig('./figures/figure3/depth_bars.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
data = pd.read_pickle('./processed_data/samples_10000_model.pkl')

# Plot the distribution of inequality across groups.
plt.figure(figsize=(8,6))
sns.barplot(data, x='age_range', y='indicator', order=['18-23','24-30','31-40','41-50','51-60','60+'], errorbar=bernoulli_ci, palette=sns.color_palette("Blues", 6), capsize=0.2, edgecolor=sns.color_palette("Blues", 6))
plt.hlines(y=data['indicator'].mean(), xmin=-0.75, xmax=5.75, lw=5, color='gray', ls='--', alpha=0.5, label='Avg')
plt.ylabel('Ratio for Deep ICs')
plt.xlabel('Age')
plt.xlim(-0.75, 5.75)
plt.legend()
plt.savefig('./figures/figure3/IC_disparity_model.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
print(data['indicator'].mean())

In [ ]:
data_em  = pd.read_pickle('./processed_data/inequality_stats.pkl')
data_model = data.groupby('age_range')['indicator'].mean().reset_index()

In [ ]:
results = []
for age in ['18-23','24-30','31-40','41-50','51-60','60+']:
    values_em = data_em.loc[data_em['age_range']==age]['indicator'].to_numpy()[0]
    print(values_em)
    values_md = data_model.loc[data_model['age_range']==age]['indicator'].to_numpy()[0]
    print(values_md)
    results.append((values_em, values_md))
results = pd.DataFrame(results,columns=['empirical','model'])

In [ ]:
plt.scatter(results['empirical'], results['model'])
plt.plot(results['empirical'],results['empirical'],)

# Obtain the calibrated parameter mappings.

In [ ]:
parameters_em = np.array([-0.01645, 0.01893, 0.35866])
parameters_th = np.array([-0.01903, 0.01541, 0.35749])

# sigma ,beta -- non-extreme cases
def get_mapping(parameters_em, parameters_th):
    sigma_mapping = parameters_em[0]/parameters_th[0]*0.0005
    beta_mapping = parameters_em[1]/parameters_th[1]*0.1
    return sigma_mapping,beta_mapping

def mapping_function_idx(em_range, md_range, em_explore_idx, em_exploit_idx):
    # em_explore, em_exploit --> 10, 10
    # md_explore, md_exploit --> 9, 13

    em_explore, em_exploit = em_range[0]-1, em_range[1]-1    
    md_explore, md_exploit = md_range[0]-1, md_range[1]-1

    return np.round(em_explore_idx * md_explore/em_explore,0).astype('int'),  np.round(em_exploit_idx * md_exploit/em_exploit, 0).astype('int')

def get_samples(overall_num):
    from collections import Counter
    age_range = ['18-23','24-30','31-40','41-50','51-60','60+']
    with open('./processed_data/age_range_all_dist.pkl','rb') as f:
        results = pickle.load(f)
    age_dist = results['age_dist']
    age_dist = np.array(age_dist)/np.sum(age_dist)
    overall_population = np.random.choice(np.arange(6), p=age_dist,size=overall_num)
    overall_population = Counter(overall_population)
    all_samples = {}
    for i in range(6):
        age_rr = age_range[i]
        cnt = overall_population[i]
        dist = results[age_range[i]]
        age_samples = np.random.choice(a=np.arange(100), p=dist.flatten(), size=cnt)
        sigma = age_samples//10 
        beta = age_samples%10
        age_samples = [z for z in zip(sigma, beta)]
        all_samples[age_rr] = {'cnt':cnt, 'samples': age_samples} 
    return all_samples



In [ ]:
samples = get_samples(10000)

In [ ]:
samples['31-40']

In [ ]:
age_range = ['18-23','24-30','31-40','41-50','51-60','60+']
results_samples = {}
for aa in age_range:
    ss = samples[aa]['samples']
    rs = []
    for s in ss:
        sigma_th, beta_th = mapping_function_idx((10,10), (10, 10), s[0], s[1])
        rs.append((sigma_th,beta_th))
    results_samples[aa] = rs

In [ ]:
with open('./processed_data/samples_results_10000.pkl','wb') as f:
    pickle.dump(results_samples, f)

In [ ]:
def bernoulli_ci(values):
    import statsmodels.api as sm
    n = len(values)
    p = values.mean()
    conf_int = sm.stats.proportion_confint(p * n, n, method='normal')
    return conf_int
data = pd.read_pickle('./processed_data/samples_10000_model.pkl')

# Plot the distribution of inequality across groups.
plt.figure(figsize=(8,6))
sns.barplot(data, x='age_range', y='indicator', order=['18-23','24-30','31-40','41-50','51-60','60+'], errorbar=bernoulli_ci, palette=sns.color_palette("Blues", 6), capsize=0.2, edgecolor=sns.color_palette("Blues", 6))
plt.hlines(y=data['indicator'].mean(), xmin=-0.75, xmax=5.75, lw=5, color='gray', ls='--', alpha=0.5, label='Avg')
plt.ylabel('Ratio for Deep ICs')
plt.xlabel('Age')
plt.xlim(-0.75, 5.75)
plt.legend()
plt.savefig('./figures/figure3/IC_disparity_model.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
with open('./processed_data/age_range_all_dist.pkl','rb') as f:
    results = pickle.load(f)
age_dist = results['age_dist']
age_dist = np.array(age_dist)/np.sum(age_dist)

In [ ]:
age_dist

In [ ]:
# Plot the distribution of inequality across groups.
age_range = ['18-23','24-30','31-40','41-50','51-60','60+']
plt.figure(figsize=(12,4))
plt.bar(np.arange(6), age_dist, color=[sns.color_palette("Blues", 10)[4],sns.color_palette("Blues", 10)[9],sns.color_palette("Greens", 10)[4], 
                                       sns.color_palette("Greens", 10)[9], sns.color_palette("YlOrBr", 10)[4], sns.color_palette("YlOrBr", 10)[9]])
plt.xticks(np.arange(6), age_range)
plt.ylabel('Proportion')
plt.xlabel('Age')
plt.xlim(-0.75, 5.75)
# plt.legend()
plt.savefig('./figures/figure3/age_dist.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
[sns.color_palette("Blues", 5)[3], sns.color_palette("Greens", 5)[3], sns.color_palette("YlOrBr", 5)[3]]